# The AI Telco Troubleshooting Challenge

Goal: Enhance the accuracy of Qwen3-32B when answering telco troubleshooting questions in telelogs data.

## I/ Set-up

In [ ]:
!git clone https://github.com/Kodro23/Telco-challenge

fatal: destination path 'Telco-challenge' already exists and is not an empty directory.


### 1. Load libraries

In [ ]:
import pandas as pd
import numpy as np

### 2.Load data

In [ ]:
#load data
df = pd.read_csv("/content/Telco-challenge/train.csv")
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/train.csv'

In [ ]:
#Function to clean questions
def clean_question(question):
    lines=question.split("\n")
    content=[]
    for line in lines:
        if "|" in line:
            content.append(line.split("|"))
    drive_test_data=[{"Observation": i+1,**dict(zip(content[0],row))} for i,row in enumerate(content[1:])]
    engineering_params=[{"Observation": i+1,**dict(zip(content[11],row))} for i,row in enumerate(content[12:])]
    #clean question
    ##recompute begining of the prompt
    q=""
    for l in [l+"\n" for l in lines[:19]]:
        q=q+l
    ##assemble
    cleaned_question= f" Question: {q} \nDrive test data: {drive_test_data} \nEngineering parameters {engineering_params} "
    return cleaned_question
#Apply to dataset
df["cleaned_question"]=df["question"].apply(lambda x: clean_question(x))
print(df["cleaned_question"][0])

## II/ Untrained model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:

# load the tokenizer and the model
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [ ]:
# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

In [ ]:
# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)